# NoSQL Databases & MongoDB Fundamentals

Relational databases (SQL) are amazing, but they have a strict rule: **Every row in a table must have the exact same columns.** But what if you are scraping data from Twitter, Reddit, and News websites? A tweet has a "retweet count", a Reddit post has "upvotes", and a news article has an "author bio". If you try to force all of these into a single SQL table, you will end up with thousands of empty (`NULL`) columns. 

This is where **NoSQL (Not Only SQL)** comes in. NoSQL databases are designed to handle unstructured or semi-structured data. They are incredibly flexible, highly scalable, and store data in a format that looks exactly like Python dictionaries (JSON).

The most popular NoSQL database in the world is **MongoDB**.

### The SQL to MongoDB Translation Guide:
* **Database** = Database
* **Table** = **Collection**
* **Row** = **Document**
* **Column** = **Field**

To practice in Python without needing to install a massive database server, we will use a library called `mongomock`, which perfectly simulates MongoDB in our computer's memory!

In [3]:
# In the real world, you would use: from pymongo import MongoClient
import mongomock
import pandas as pd

# 1. Connect to our temporary in-memory MongoDB server
client = mongomock.MongoClient()

# 2. Create a Database called 'CompanyDB'
db = client.CompanyDB

# 3. Create a Collection (Table) called 'employees'
collection = db.employees

print("✅ MongoDB Sandbox ready! Database and Collection created.")

✅ MongoDB Sandbox ready! Database and Collection created.


# 1. Documents and Flexible Schema (Create)
In MongoDB, data is stored as **Documents** (which are essentially just JSON objects or Python dictionaries). 

The greatest power of MongoDB is its **Flexible Schema**. Two documents in the exact same collection do not need to have the same fields!

In [4]:
# Let's insert two employees. Notice how their data structures are completely different!

employee_1 = {
    "name": "Alice",
    "age": 28,
    "department": "Engineering",
    "skills": ["Python", "SQL", "AWS"] # We can store lists directly inside a field!
}

employee_2 = {
    "name": "Bob",
    "department": "Sales",
    "monthly_sales_target": 50000,
    "contact": { # We can store dictionaries inside dictionaries!
        "phone": "555-0100",
        "email": "bob@sales.com"
    }
}

# Insert them into the collection
collection.insert_one(employee_1)
collection.insert_one(employee_2)

print("✅ Flexible Documents inserted successfully!")

✅ Flexible Documents inserted successfully!


# 2. Querying Data (Read)
Because we aren't using SQL, we don't write `SELECT * FROM table`. Instead, we use MongoDB's query syntax, which is also written as a Python dictionary!

In [5]:
# Find ALL documents in the collection
print("--- All Employees ---")
all_docs = list(collection.find())
for doc in all_docs:
    print(doc)

# Find a specific document (e.g., WHERE name = 'Alice')
print("\n--- Find Alice ---")
alice_doc = collection.find_one({"name": "Alice"})
print(alice_doc)

--- All Employees ---
{'name': 'Alice', 'age': 28, 'department': 'Engineering', 'skills': ['Python', 'SQL', 'AWS'], '_id': ObjectId('69d9234f8a1ed0f6e837dfc5')}
{'name': 'Bob', 'department': 'Sales', 'monthly_sales_target': 50000, 'contact': {'phone': '555-0100', 'email': 'bob@sales.com'}, '_id': ObjectId('69d9234f8a1ed0f6e837dfc6')}

--- Find Alice ---
{'name': 'Alice', 'age': 28, 'department': 'Engineering', 'skills': ['Python', 'SQL', 'AWS'], '_id': ObjectId('69d9234f8a1ed0f6e837dfc5')}


*(Notice the `_id` field? MongoDB automatically generates a totally unique ID for every single document you insert, acting as a built-in Primary Key!)*

# 3. Advanced Filtering (Operators)
If you want to do math filters (like `WHERE age > 25`), MongoDB uses special operators that start with a dollar sign `$`.
* `$gt`: Greater Than
* `$lt`: Less Than
* `$in`: Exists within a list

In [6]:
# Add one more employee to make it interesting
collection.insert_one({"name": "Charlie", "age": 35, "department": "Engineering"})

# Query: Find everyone older than 30
query_older = {"age": {"$gt": 30}}
results = list(collection.find(query_older))

print("--- Employees older than 30 ---")
for doc in results:
    print(doc)

--- Employees older than 30 ---
{'name': 'Charlie', 'age': 35, 'department': 'Engineering', '_id': ObjectId('69d923508a1ed0f6e837dfc7')}


# 4. Updating Data (Update)
Updating data requires two dictionaries: 
1. The **Filter**: Who do we want to update?
2. The **Update Action**: What are we changing? (We use the `$set` operator to modify fields).

In [7]:
# Alice just had a birthday! Let's update her age to 29.
filter_query = {"name": "Alice"}
update_action = {"$set": {"age": 29}}

collection.update_one(filter_query, update_action)

print("✅ Alice's age updated!")
print(collection.find_one({"name": "Alice"}))

✅ Alice's age updated!
{'name': 'Alice', 'age': 29, 'department': 'Engineering', 'skills': ['Python', 'SQL', 'AWS'], '_id': ObjectId('69d9234f8a1ed0f6e837dfc5')}


# 5. Bridging MongoDB to Pandas
As a Data Scientist, you will extract data from MongoDB and pull it into Pandas to analyze it. Because MongoDB documents are just dictionaries, Pandas handles them beautifully.

In [8]:
# Pull all Engineering department documents into a Pandas DataFrame
engineering_docs = list(collection.find({"department": "Engineering"}))

df_engineers = pd.DataFrame(engineering_docs)

print("--- MongoDB Data in a Pandas DataFrame ---")
display(df_engineers)

# Clean up our sandbox
client.drop_database('CompanyDB')

--- MongoDB Data in a Pandas DataFrame ---


,name,age,department,skills,_id
0,Alice,29,Engineering,"[Python, SQL, AWS]",69d9234f8a1ed0f6e837dfc5
1,Charlie,35,Engineering,NaN,69d923508a1ed0f6e837dfc7


*(Notice what Pandas does with Bob's missing fields? If a document doesn't have a specific field, Pandas automatically fills it with `NaN` (Not a Number)!)*

## Real-World Use Case or Analogy:
Think of SQL vs. NoSQL like **Filing Cabinets vs. Storage Bins**:

* **SQL (The Filing Cabinet)**: Before you can put a single piece of paper into the cabinet, you must buy specific folders, label them perfectly, and declare exactly what goes in them. If you buy a "Tax Forms" cabinet, every piece of paper inside must be a tax form. If you try to shove a "Recipe" in there, the cabinet locks up and rejects it. It is incredibly organized, but highly rigid.
* **NoSQL / MongoDB (The Storage Bin)**: You buy a giant plastic storage bin labeled "My Life". You toss in a tax form. Then you toss in a recipe. Then you toss in a pair of winter boots (a nested list of data). The bin doesn't care; it accepts whatever you throw at it. When you need to find something, you just tell the bin: *"Give me everything that has the word 'Tax' on it,"* and it quickly hands you the tax forms, completely ignoring the boots. It is flexible, fast, and handles the chaos of the real world effortlessly.

---